# Embedding
Embedding refers to the process of representing data, such as words, sentences, or images, in a continuous vector space. These vector representations capture semantic or contextual relationships, enabling machines to process and analyze the data more effectively. For example, word embeddings like Word2Vec or GloVe map words to high-dimensional vectors where similar words are closer in the vector space.

![embedding](./resources/embedding.png)

In [26]:
import torch

example = "Hello world this is an example"
n = 10
embedding = []
for word in example.split():
    embedding.append(torch.rand(10))

embedding

[tensor([0.5292, 0.2637, 0.0196, 0.3650, 0.5590, 0.2216, 0.7433, 0.2602, 0.8475,
         0.2328]),
 tensor([0.3413, 0.0653, 0.4282, 0.0349, 0.4430, 0.6565, 0.7985, 0.1646, 0.7156,
         0.2187]),
 tensor([0.3727, 0.2806, 0.3222, 0.1863, 0.7059, 0.6205, 0.5187, 0.7995, 0.6802,
         0.8730]),
 tensor([0.2454, 0.5350, 0.2213, 0.4116, 0.4074, 0.1918, 0.8303, 0.4199, 0.1987,
         0.8329]),
 tensor([0.1148, 0.4485, 0.7069, 0.4561, 0.6209, 0.1982, 0.3698, 0.7625, 0.2629,
         0.8508]),
 tensor([0.5915, 0.8211, 0.9920, 0.2594, 0.5300, 0.8855, 0.8479, 0.4579, 0.3067,
         0.6562])]

So we just created a simple embedding, assume that our tokenizer breaks down a sentence into words, our embedding simply assigns n dimensional vectors to the tokens.

The following example is from https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html#an-example-n-gram-language-modeling. It provides a brief idea of how an embedding can be trained using N-gram language modelling. For every token in the training data, we create pairs of N previous tokens with the current target token. For example, when N is 2
- `(['forty', 'When'], 'winters')` means: `When forty _` should return `winters`
- `(['winters', 'forty'], 'shall')` means: `forty winters _` should return `shall`
- `(['shall', 'winters'], 'besiege')` means: `winters shall _` should return `besiege`
- etc.

In [27]:
from torch import nn, optim
import torch.nn.functional as F

CONTEXT_SIZE = 2
EMBEDDING_DIM = 10

# We will use Shakespeare Sonnet 2
test_sentence = """When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a totter'd weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say, within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserv'd thy beauty's use,
If thou couldst answer 'This fair child of mine
Shall sum my count, and make my old excuse,'
Proving his beauty by succession thine!
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold.""".split()
# we should tokenize the input, but we will ignore that for now
# build a list of tuples.
# Each tuple is ([ word_i-CONTEXT_SIZE, ..., word_i-1 ], target word)
ngrams = [
    (
        [test_sentence[i - j - 1] for j in range(CONTEXT_SIZE)],
        test_sentence[i]
    )
    for i in range(CONTEXT_SIZE, len(test_sentence))
]
# Print the first 3, just so you can see what they look like.
print(ngrams[:3])

vocab = set(test_sentence)
# Word to index, given a word, return its ID (index)
word_to_ix = {word: i for i, word in enumerate(vocab)}
print(list(word_to_ix.items())[:3])


class NGramLanguageModeler(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        super(NGramLanguageModeler, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((1, -1))
        out = F.relu(self.linear1(embeds))
        out = self.linear2(out)
        log_probs = F.log_softmax(out, dim=1)
        return log_probs


losses = []
loss_function = nn.NLLLoss()
model = NGramLanguageModeler(len(vocab), EMBEDDING_DIM, CONTEXT_SIZE)
optimizer = optim.SGD(model.parameters(), lr=0.001)

for epoch in range(10):
    total_loss = 0
    for context, target in ngrams:

        # Step 1. Prepare the inputs to be passed to the model (i.e, turn the words
        # into integer indices and wrap them in tensors)
        context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)

        # Step 2. Recall that torch *accumulates* gradients. Before passing in a
        # new instance, you need to zero out the gradients from the old
        # instance
        model.zero_grad()

        # Step 3. Run the forward pass, getting log probabilities over next
        # words
        log_probs = model(context_idxs)

        # Step 4. Compute your loss function. (Again, Torch wants the target
        # word wrapped in a tensor)
        loss = loss_function(log_probs, torch.tensor([word_to_ix[target]], dtype=torch.long))

        # Step 5. Do the backward pass and update the gradient
        loss.backward()
        optimizer.step()

        # Get the Python number from a 1-element Tensor by calling tensor.item()
        total_loss += loss.item()
    losses.append(total_loss)
print(losses)  # The loss decreased every iteration over the training data!

# To get the embedding of a particular word, e.g. "beauty"
print(model.embeddings.weight[word_to_ix["beauty"]])

[(['forty', 'When'], 'winters'), (['winters', 'forty'], 'shall'), (['shall', 'winters'], 'besiege')]
[('trenches', 0), ('thine', 1), ('lies,', 2)]
[523.3565304279327, 520.7445299625397, 518.153026342392, 515.5800788402557, 513.0235071182251, 510.48327350616455, 507.9597611427307, 505.4508972167969, 502.95450091362, 500.47041606903076]
tensor([ 0.0199, -1.8107,  0.1332, -0.3536,  0.1495,  0.8356,  1.7264, -1.7492,
         2.2223,  0.0736], grad_fn=<SelectBackward0>)


Following is an exercise from https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html#exercise-computing-word-embeddings-continuous-bag-of-words, to implement the embedding training part using Continuous Bag-of-Words (CBOW).

In [28]:
from torch import nn, optim
import torch.nn.functional as F

CONTEXT_SIZE = 2  # 2 words to the left, 2 to the right
raw_text = """We are about to study the idea of a computational process.
Computational processes are abstract beings that inhabit computers.
As they evolve, processes manipulate other abstract things called data.
The evolution of a process is directed by a pattern of rules
called a program. People create programs to direct processes. In effect,
we conjure the spirits of the computer with our spells.""".split()

# By deriving a set from `raw_text`, we deduplicate the array
vocab = set(raw_text)
vocab_size = len(vocab)

word_to_ix = {word: i for i, word in enumerate(vocab)}
data = []
for i in range(CONTEXT_SIZE, len(raw_text) - CONTEXT_SIZE):
    context = (
        [raw_text[i - j - 1] for j in range(CONTEXT_SIZE)]
        + [raw_text[i + j + 1] for j in range(CONTEXT_SIZE)]
    )
    target = raw_text[i]
    data.append((context, target))
print(data[:5])


class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size):
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Because the window size is 2, we need to input 4 words
        self.linear1 = nn.Linear(context_size * 2 * embedding_dim, 128)
        # Output is the size of the vocabulary
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((1, -1))
        out = F.relu(self.linear1(embeds))
        out = self.linear2(out)
        log_probs = F.log_softmax(out, dim=1)
        return log_probs

# Create your model and train. Here are some functions to help you make
# the data ready for use by your module.


def make_context_vector(context, word_to_ix):
    idxs = [word_to_ix[w] for w in context]
    return torch.tensor(idxs, dtype=torch.long)


losses = []
loss_function = nn.NLLLoss()
model = CBOW(len(vocab), EMBEDDING_DIM, CONTEXT_SIZE)
optimizer = optim.SGD(model.parameters(), lr=0.001)

for epoch in range(20):
    total_loss = 0
    for context, target in data:
        context_idxs = make_context_vector(context, word_to_ix)
        model.zero_grad()
        log_probs = model(context_idxs)
        loss = loss_function(log_probs, torch.tensor([word_to_ix[target]], dtype=torch.long))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    losses.append(total_loss)
print(losses)  # The loss decreased every iteration over the training data!
# To get the embedding of a particular word, e.g. "processes"
print(model.embeddings.weight[word_to_ix["processes"]])


[(['are', 'We', 'to', 'study'], 'about'), (['about', 'are', 'study', 'the'], 'to'), (['to', 'about', 'the', 'idea'], 'study'), (['study', 'to', 'idea', 'of'], 'the'), (['the', 'study', 'of', 'a'], 'idea')]
[228.62316060066223, 227.1999638080597, 225.7860209941864, 224.37961387634277, 222.98124146461487, 221.59011125564575, 220.2066776752472, 218.82924580574036, 217.45870065689087, 216.09294891357422, 214.73311352729797, 213.37529802322388, 212.0224916934967, 210.6728925704956, 209.32611966133118, 207.98115181922913, 206.6392149925232, 205.29902362823486, 203.96137690544128, 202.62577176094055]
tensor([ 0.2043,  1.3175,  1.4366,  1.1881, -0.0753, -1.3475, -0.4166, -0.2149,
        -0.8751, -0.0586], grad_fn=<SelectBackward0>)
